# IPL First Innings Score Prediction

Same workflow as the house price notebook: clean and prepare the data, get a baseline regression score, then apply cross-validation, model comparison and hyperparameter tuning to improve performance.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

## Loading the data

In [2]:
df = pd.read_csv("ipl.csv")
df.shape

(76014, 15)

In [3]:
df.head()

,mid,date,venue,bat_team,bowl_team,batsman,bowler,runs,wickets,overs,runs_last_5,wickets_last_5,striker,non-striker,total
0,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,P Kumar,1,0,0.1,1,0,0,0,222
1,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,1,0,0.2,1,0,0,0,222
2,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.2,2,0,0,0,222
3,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.3,2,0,0,0,222
4,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.4,2,0,0,0,222


In [4]:
df.isnull().sum()

mid               0
date              0
venue             0
bat_team          0
bowl_team         0
batsman           0
bowler            0
runs              0
wickets           0
overs             0
runs_last_5       0
wickets_last_5    0
striker           0
non-striker       0
total             0
dtype: int64

In [5]:
df['bat_team'].unique()

<StringArray>
[      'Kolkata Knight Riders',         'Chennai Super Kings',
            'Rajasthan Royals',              'Mumbai Indians',
             'Deccan Chargers',             'Kings XI Punjab',
 'Royal Challengers Bangalore',            'Delhi Daredevils',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',     'Rising Pune Supergiants',
               'Gujarat Lions',      'Rising Pune Supergiant']
Length: 14, dtype: str

## Cleaning the data

Only teams that played consistently across seasons are kept, since teams that only appear for a season or two don't give the model enough to learn from. The first five overs of an innings are also dropped, because the score at that point is too volatile to be a useful predictor.

In [6]:
consistent_teams = [
    'Kolkata Knight Riders', 'Chennai Super Kings', 'Rajasthan Royals',
    'Mumbai Indians', 'Kings XI Punjab', 'Royal Challengers Bangalore',
    'Delhi Daredevils', 'Sunrisers Hyderabad'
]

df = df[(df['bat_team'].isin(consistent_teams)) & (df['bowl_team'].isin(consistent_teams))]
df = df[df['overs'] >= 5.0]
df.shape

(40108, 15)

In [7]:
df = df.drop(['mid', 'date', 'venue', 'batsman', 'bowler'], axis=1)
df.head()

,bat_team,bowl_team,runs,wickets,overs,runs_last_5,wickets_last_5,striker,non-striker,total
32,Kolkata Knight Riders,Royal Challengers Bangalore,61,0,5.1,59,0,41,10,222
33,Kolkata Knight Riders,Royal Challengers Bangalore,61,1,5.2,59,1,41,10,222
34,Kolkata Knight Riders,Royal Challengers Bangalore,61,1,5.3,59,1,41,0,222
35,Kolkata Knight Riders,Royal Challengers Bangalore,61,1,5.4,59,1,41,0,222
36,Kolkata Knight Riders,Royal Challengers Bangalore,61,1,5.5,58,1,41,0,222


## Encoding categorical features

In [8]:
df = pd.get_dummies(df, columns=['bat_team', 'bowl_team'])
df.shape

(40108, 24)

## Baseline model

In [9]:
X = df.drop('total', axis=1)
y = df['total']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
baseline_score = baseline_model.score(X_test, y_test)
print(f"Baseline Linear Regression R2: {baseline_score:.4f}")

Baseline Linear Regression R2: 0.6681


## Improving performance

A plain linear model struggles here because the relationship between overs, wickets and the final total is far from linear. Comparing a few algorithms with cross-validation makes that obvious.

In [11]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100),
    'XGBoost': XGBRegressor(random_state=42)
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Linear Regression: 0.6234 (+/- 0.0563)
Ridge: 0.6234 (+/- 0.0563)
Decision Tree: 0.2476 (+/- 0.1086)
Random Forest: 0.5302 (+/- 0.0706)
XGBoost: 0.5081 (+/- 0.0820)


XGBoost has the most room to grow once its hyperparameters are tuned, since a shallow default tree depth is holding it back on a dataset this size.

In [12]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(f"Best CV R2: {grid_search.best_score_:.4f}")

{'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200}
Best CV R2: 0.9123


In [13]:
best_model = grid_search.best_estimator_
final_score = best_model.score(X_test, y_test)

print(f"Baseline Linear Regression R2: {baseline_score:.4f}")
print(f"Tuned XGBoost R2: {final_score:.4f}")
print(f"Improvement: {(final_score - baseline_score) * 100:.2f} points")

Baseline Linear Regression R2: 0.6681
Tuned XGBoost R2: 0.9245
Improvement: 25.64 points


## Conclusion

Moving from a plain linear regression to a tuned XGBoost model, guided by cross-validation and a grid search over the key hyperparameters, gives a large jump in R2, since a gradient-boosted tree can capture the non-linear way overs, wickets and run rate interact to determine the final score.